# Exercise 7.2: Sampling calorimeter

From *Programming in High Energy Particle Physics*, Chapter 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch07/ex07_2.ipynb)

Exercise 7.2 Sampling calorimeter Simulate a sampling calorimeter with alternating layers of lead (absorber) and plastic scintillator (active medium). Vary the absorber-to-scintillator ratio and study how the energy resolution (\(\sigma_E/E\)) and sampling fraction change.

<details><summary>Hint</summary>

Record the energy in the scintillator and in the lead separately. The sampling fraction is \(E_\text{scint}/(E_\text{scint}+E_\text{Pb})\). Fit the response at several beam energies with Eq. 7.3. A useful empirical rule for electromagnetic sampling calorimeters is \(a_s \approx 2.7\%\sqrt{d/f_\text{samp}}\), with \(d\) the thickness of the active layers in mm and \(f_\text{samp}\) the sampling fraction for minimum-ionizing particles.

</details>

### Setup

The Colab analysis is an **empirical stand-in**, not Geant4 shower transport. `geant4_pybind` is an optional heavy download, including large datasets; a full C++ Geant4 build is required for the simulation below.

In [ ]:
import sys
IN_COLAB="google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scipy matplotlib
INSTALL_GEANT4_BINDINGS=False
if IN_COLAB and INSTALL_GEANT4_BINDINGS:
    %pip install -q geant4_pybind
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)

### Complete local Geant4 shower simulation

The code records active and absorber energy separately. It varies Pb layer thickness while holding the total Pb near 120 mm (about 21 X₀), so changes in shower containment are reduced; the number of layers and calorimeter length change.

In [ ]:
%%writefile ex07_2.cc
#include "G4RunManager.hh"
#include "G4VUserDetectorConstruction.hh"
#include "G4VUserPrimaryGeneratorAction.hh"
#include "G4VUserActionInitialization.hh"
#include "G4UserEventAction.hh"
#include "G4UserSteppingAction.hh"
#include "G4NistManager.hh"
#include "G4Box.hh"
#include "G4LogicalVolume.hh"
#include "G4PVPlacement.hh"
#include "G4ParticleGun.hh"
#include "G4ParticleTable.hh"
#include "G4SystemOfUnits.hh"
#include "G4Event.hh"
#include "G4Step.hh"
#include "FTFP_BERT.hh"
#include "Randomize.hh"
#include <fstream>
#include <string>
#include <cstdlib>
class Calorimeter : public G4VUserDetectorConstruction {
  double leadMM;
public:
  G4LogicalVolume *lead=nullptr,*scint=nullptr;
  double startZ=0;
  explicit Calorimeter(double d):leadMM(d){}
  G4VPhysicalVolume* Construct() override {
    auto* nist=G4NistManager::Instance();
    auto* air=nist->FindOrBuildMaterial("G4_AIR");
    auto* pb=nist->FindOrBuildMaterial("G4_Pb");
    auto* plastic=nist->FindOrBuildMaterial("G4_POLYSTYRENE");
    auto* world=new G4LogicalVolume(new G4Box("world",50*cm,50*cm,1*m),air,"world");
    auto* pv=new G4PVPlacement(nullptr,{},world,"world",nullptr,false,0);
    const int n=static_cast<int>(120.0/leadMM); // same ~120 mm Pb = ~21 X0
    const double scintMM=2.0;
    const double pitch=(leadMM+scintMM)*mm;
    startZ=-0.5*n*pitch;
    lead=new G4LogicalVolume(new G4Box("Pb",5*cm,5*cm,0.5*leadMM*mm),pb,"Pb");
    scint=new G4LogicalVolume(new G4Box("Scint",5*cm,5*cm,0.5*scintMM*mm),plastic,"Scint");
    for(int i=0;i<n;++i) {
      new G4PVPlacement(nullptr,G4ThreeVector(0,0,startZ+i*pitch+0.5*leadMM*mm),lead,"Pb",world,false,i);
      new G4PVPlacement(nullptr,G4ThreeVector(0,0,startZ+i*pitch+(leadMM+0.5*scintMM)*mm),scint,"Scint",world,false,i);
    }
    return pv;
  }
};
class ElectronBeam : public G4VUserPrimaryGeneratorAction {
  G4ParticleGun gun{1};
public:
  ElectronBeam(double energyGeV,double startZ) {
    gun.SetParticleDefinition(G4ParticleTable::GetParticleTable()->FindParticle("e-"));
    gun.SetParticleEnergy(energyGeV*GeV);
    gun.SetParticlePosition(G4ThreeVector(0,0,startZ-5*mm));
    gun.SetParticleMomentumDirection(G4ThreeVector(0,0,1));
  }
  void GeneratePrimaries(G4Event* e) override {gun.GeneratePrimaryVertex(e);}
};
class CalEvent : public G4UserEventAction {
  std::ofstream out;
  double eScint=0,ePb=0;
public:
  explicit CalEvent(const std::string& path):out(path) {out << "scint_MeV,pb_MeV\n";}
  void BeginOfEventAction(const G4Event*) override {eScint=ePb=0;}
  void AddScint(double e){eScint+=e;}
  void AddPb(double e){ePb+=e;}
  void EndOfEventAction(const G4Event*) override {out << eScint/MeV << ',' << ePb/MeV << '\n';}
};
class CalStep : public G4UserSteppingAction {
  const Calorimeter* det;
  CalEvent* evt;
public:
  CalStep(const Calorimeter* d,CalEvent* e):det(d),evt(e){}
  void UserSteppingAction(const G4Step* step) override {
    auto* lv=step->GetPreStepPoint()->GetTouchableHandle()->GetVolume()->GetLogicalVolume();
    if(lv==det->scint) evt->AddScint(step->GetTotalEnergyDeposit());
    if(lv==det->lead) evt->AddPb(step->GetTotalEnergyDeposit());
  }
};
class Actions : public G4VUserActionInitialization {
  const Calorimeter* det;
  double energy;
  std::string path;
public:
  Actions(const Calorimeter* d,double e,std::string p):det(d),energy(e),path(p){}
  void Build() const override {
    SetUserAction(new ElectronBeam(energy,det->startZ));
    auto* evt=new CalEvent(path);SetUserAction(evt);
    SetUserAction(new CalStep(det,evt));
  }
};
int main(int argc,char** argv) {
  if(argc<5) return 2; // Pb thickness mm, energy GeV, events, CSV
  const double leadMM=std::atof(argv[1]);
  if(leadMM<=0 || leadMM>5) return 3;
  G4Random::setTheSeed(42);
  auto* rm=new G4RunManager;
  auto* det=new Calorimeter(leadMM);
  rm->SetUserInitialization(det);
  rm->SetUserInitialization(new FTFP_BERT);
  rm->SetUserInitialization(new Actions(det,std::atof(argv[2]),argv[4]));
  rm->Initialize();rm->BeamOn(std::atoi(argv[3]));delete rm;
}


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ex07_2 LANGUAGES CXX)
find_package(Geant4 REQUIRED)
include(${Geant4_USE_FILE})
add_executable(ex07_2 ex07_2.cc)
target_link_libraries(ex07_2 ${Geant4_LIBRARIES})


Local commands after extracting both files:

```bash
cmake -S . -B build
cmake --build build -j2
./build/ex07_2 1 1 500 pb1_E1.csv
./build/ex07_2 2 1 500 pb2_E1.csv
./build/ex07_2 4 1 500 pb4_E1.csv
```

Repeat at beam energies 2, 5 and 10 GeV; compare `scint_MeV/(scint_MeV+pb_MeV)` and the standard deviation of calibrated scintillator energy divided by its mean. Geant4 datasets must be configured locally. The empirical Colab calculation below runs without Geant4.
If CMake cannot locate an optional dependency in an existing Geant4 installation, a direct build is also possible with `c++ ex07_N.cc -o ex07_N $(geant4-config --cflags --libs)` (replace `N` with this exercise number and configure the library path for that installation).

### Colab stand-in exercise

In [ ]:
lead_mm=np.array([1.,2.,4.])
scint_mm=2.0
# TODO: estimate MIP sampling fractions from layer thickness and stopping powers.
# TODO: use a_s = 0.027 sqrt(d/f_samp) and generate responses at 1, 2, 5, 10 GeV.
f_samp=None

In [ ]:
assert np.all(lead_mm>0)
print("Self-check: thinner Pb should increase sampling fraction and improve the stochastic term.")

Plot sampling fraction and σE/E with labelled axes. Explain why the empirical stand-in cannot replace the Geant4 shower simulation.